# Federated Multivariate Linear Regression

**One of three method-focused notebooks for the BioITWorld / ISMB talk.**

This notebook does exactly one thing: fit a multivariate linear regression
to predict log(C-peptide AUC) from autoantibody and demographic features,
*without ever pooling raw data across institutions*. Each institution fits
on its own subjects; only the coefficient vectors are exchanged.

## The federation problem

Each SDY study comes from a different research network and lives behind a
different institutional firewall. Pooling raw subject-level data across
sites is rarely possible — IRB review, data-use agreements, and BAA
constraints typically prohibit it. The federated alternative is:

> Each site fits its own model on its own data. The sites exchange only
> model parameters. A coordinator combines those parameters into a single
> global model that all sites can use.

For linear regression, the natural aggregation rule is the
**sample-size-weighted average** of per-institution coefficients,
sometimes called **FedAvg-by-N** in the literature:

$$
\hat\beta_\mathrm{fed} = \sum_k \frac{N_k}{N_\mathrm{total}}\,\hat\beta_k
$$

This recovers the centralised OLS solution **exactly** when the per-site
feature matrices share the same column means and the residual variances
agree — both reasonable when sites measure the same biology. When they
don't, FedAvg-by-N stays well-behaved (it never blows up) and a sample-size
weighting prevents tiny sites from dominating.

## Data panels in this notebook

- **Panel A** — the legacy 9-feature panel (5 autoantibodies + 3 age-group
  dummies + Sex) across 4 studies, N=150. Matches the inputs used in the
  earlier CNN work.
- **Panel B** — the Jeff-extended panel (15 features after one-hot
  encoding of race / ethnicity / treatment cohort, plus BMI, height,
  weight, exact age, disease duration since T1D diagnosis) across 3
  studies, N=98. SDY797 and SDY1625 are not in this panel because no Jeff
  data was supplied for them.

The federated MLR is run on both panels so the talk can contrast what a
linear model recovers from the legacy autoantibody-only panel versus the
extended biological panel.


## 1. Setup

In [ ]:
from __future__ import annotations
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Locate the repo root and add src/ to the path for the data loader
REPO = Path.cwd()
if (REPO / "src").exists():
    sys.path.insert(0, str(REPO / "src"))
elif (REPO.parent / "src").exists():
    REPO = REPO.parent
    sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from scipy.stats import f as fdist, ncf

import oadr_data as od

RNG_SEED = 42
N_FOLDS = 5
ALPHA_STAT = 0.05    # significance level for power calculations
np.random.seed(RNG_SEED)

(REPO / "results").mkdir(exist_ok=True)
(REPO / "figures").mkdir(exist_ok=True)
print(f"Repo root: {REPO}")


### A note on `src/oadr_data.py`

The cell above imports the project's single shared utility, which handles
all the per-study quirks (column-name typos, `IA_2ic` ↔ `IA2IC` antibody
naming, the per-study autoantibody panels that don't all measure the same
five antibodies, the Jeff-extended panel merge with disease-duration
computation). Open `src/oadr_data.py` if you want to read the cleanup line
by line. Everything else in this notebook is intended to be readable
top-to-bottom without chasing imports.


In [ ]:
# Load both panels and report their shape and per-study composition
A = od.load_panel_a_all()
Xa = A[od.PANEL_A_FEATURES].values.astype(float)
ya = A[od.PANEL_A_TARGET].values.astype(float)
studies_a = A["Study"].values
fa = list(od.PANEL_A_FEATURES)

B = od.load_panel_b_all()
Xb_df, yb_s, fb = od.panel_b_design_matrix(B)
Xb = Xb_df.values.astype(float)
yb = yb_s.values.astype(float)
studies_b = B["Study"].values
fb = list(fb)

print(f"Panel A: N={len(A)}, features={len(fa)}")
print(A.groupby("Study").size().to_string())
print()
print(f"Panel B: N={len(B)}, features={len(fb)}")
print(B.groupby("Study").size().to_string())


## 2. Power calculation utility

For every fitted model below we'll report:

- **R²** on out-of-fold predictions (out-of-fold means: every subject's
  prediction comes from a model that did not see them during training).
- **Cohen's f²** = R² / (1 − R²): a unitless effect-size convention used
  in F-test power calculations.
- **Achieved power**: the probability of correctly rejecting "the model
  found nothing" given the effect size we actually observed at this N
  with this many predictors, computed from a non-central F distribution
  via `scipy.stats.ncf`.

The `bin/test_power_analysis.py` script that previously lived in this
repo wrapped `statsmodels.stats.power.FTestPower.solve_power` in a way
that returned wrong numbers (≈ 5% power for R² = 0.15 at N = 154 instead
of the correct ≈ 97%). We compute power directly from the non-central F
here.


In [ ]:
def calc_power(n, k, f2, alpha=ALPHA_STAT):
    """A-priori F-test power for multiple regression at sample size n, k predictors,
    Cohen's f-squared effect size f2.  Returns NaN if the design is degenerate."""
    if n <= k + 1 or f2 <= 0:
        return float("nan")
    df_num, df_denom = k, n - k - 1
    F_crit = fdist.ppf(1 - alpha, df_num, df_denom)
    nc = f2 * n               # non-centrality parameter
    return float(1 - ncf.cdf(F_crit, df_num, df_denom, nc))


def f2_from_r2(r2):
    """Cohen's f² from R².  Returns 0 outside the (0, 1) range — useful as
    a sentinel: if R² is negative (model worse than predicting the mean),
    the effect size is zero and downstream power is also zero."""
    return r2 / (1 - r2) if 0 < r2 < 1 else 0.0


def metrics(y_true, y_pred):
    """MSE, R², and the number of non-NaN positions in y_pred."""
    mask = ~np.isnan(y_pred)
    if mask.sum() < 2:
        return float("nan"), float("nan"), int(mask.sum())
    mse = float(np.mean((y_true[mask] - y_pred[mask]) ** 2))
    rss = float(np.sum((y_true[mask] - y_pred[mask]) ** 2))
    tss = float(np.sum((y_true[mask] - y_true[mask].mean()) ** 2))
    r2 = 1 - rss / tss if tss > 0 else float("nan")
    return mse, r2, int(mask.sum())


## 3. The federated MLR algorithm, written out long-hand

Three loops:

1. **Cross-validation outer loop**: split subjects into 5 folds, stratified
   by `Study` so every fold contains the same proportion of every
   institution's subjects.
2. **Per-institution inner loop**: each institution fits its own
   `LinearRegression` on its own training subjects, using a per-institution
   `MinMaxScaler` so feature ranges stay strictly local. Subject-level
   data never crosses the wire.
3. **Aggregation**: the coordinator receives one `(coef_, intercept_, N_k)`
   triple per institution and computes the sample-size-weighted average.

At prediction time, each held-out subject is scaled with **their own
institution's scaler** (which the institution kept) and then run through
the aggregated coefficient vector. This keeps the prediction step as
privacy-preserving as the training step — nothing about other institutions'
feature ranges is needed.


In [ ]:
def federated_mlr_oof(X, y, studies, n_splits=N_FOLDS, seed=RNG_SEED):
    """Out-of-fold predictions under federated MLR with FedAvg-by-N aggregation.

    Returns an array `oof` of the same length as y; `oof[i]` is the prediction
    for subject i made by a model that did not see them during training.
    """
    X = np.asarray(X); y = np.asarray(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.full_like(y, np.nan, dtype=float)

    for fold_idx, (tr, te) in enumerate(skf.split(X, studies)):
        # 1. Per-institution local fits
        local_models = {}   # study_id  -> (coef_, intercept_, N_k)
        scalers = {}        # study_id  -> MinMaxScaler fit on that institution's training data
        for s in sorted(set(studies[tr])):
            idx = tr[studies[tr] == s]
            if len(idx) < 2:   # safety — degenerate per-study fold
                continue
            sc = MinMaxScaler().fit(X[idx])
            scalers[s] = sc
            m = LinearRegression().fit(sc.transform(X[idx]), y[idx])
            local_models[s] = (m.coef_, m.intercept_, len(idx))

        # 2. Coordinator aggregates: sample-size-weighted mean of per-site coefficients
        coefs = np.stack([local_models[s][0] for s in local_models])
        intercepts = np.array([local_models[s][1] for s in local_models])
        sizes = np.array([local_models[s][2] for s in local_models])
        agg_coef = np.average(coefs, axis=0, weights=sizes)
        agg_intercept = np.average(intercepts, weights=sizes)

        # 3. Predict each held-out subject with their own institution's scaler
        for i in te:
            s = studies[i]
            if s not in scalers:
                continue
            xs = scalers[s].transform(X[i:i+1])[0]
            oof[i] = agg_coef @ xs + agg_intercept

    return oof


## 4. Run it on Panel A and Panel B

We evaluate four configurations:

| Configuration | Panel | Studies | N |
|---|---|---|---|
| Panel A, all 4 studies | A | SDY524 + SDY569 + SDY797 + SDY1737 | 150 |
| Panel A, drop SDY1737 | A | SDY524 + SDY569 + SDY797 | 134 |
| Panel B, all 3 studies | B | SDY524 + SDY569 + SDY1737 | 98 |
| Panel B, drop SDY1737 | B | SDY524 + SDY569 | 82 |

Dropping SDY1737 is a sensitivity check — that study contributes only 16
subjects and a federated method that depends critically on it would be
suspect.


In [ ]:
configs = [
    ("Panel A, all 4 studies", Xa, ya, studies_a, fa),
    ("Panel A, drop SDY1737",  Xa[studies_a != "SDY1737"],
                              ya[studies_a != "SDY1737"],
                              studies_a[studies_a != "SDY1737"],
                              fa),
    ("Panel B, all 3 studies", Xb, yb, studies_b, fb),
    ("Panel B, drop SDY1737",  Xb[studies_b != "SDY1737"],
                              yb[studies_b != "SDY1737"],
                              studies_b[studies_b != "SDY1737"],
                              fb),
]

rows = []
for name, X, y, st, fnames in configs:
    oof = federated_mlr_oof(X, y, st)
    mse, r2, n = metrics(y, oof)
    f2 = f2_from_r2(r2)
    power = calc_power(n, len(fnames), f2)
    rows.append({"config": name, "N": n, "k": len(fnames),
                 "MSE": mse, "R2": r2, "Cohen_f2": f2,
                 "achieved_power": power})
    print(f"{name:30s}  N={n:3d}  MSE={mse:.3f}  R²={r2:+.3f}  f²={f2:.3f}  power={power:.3f}")

mlr_results = pd.DataFrame(rows)
mlr_results.to_csv("results/federated_mlr_results.csv", index=False)


## 5. SDY569-alone vs federated scatter

The single most legible figure for this talk: each held-out subject is
plotted as observed log(C-peptide AUC) on the x-axis against the model's
prediction on the y-axis. Two panels:

- **Left**: predictions for SDY569 subjects, fit on SDY569 alone. SDY569
  has only 10 subjects total; this is what the smallest institution can
  do in isolation.
- **Right**: predictions for every subject, fit federatedly across all
  institutions. The improvement over the left panel is the federation's
  payoff.


In [ ]:
def mlr_solo_vs_federated_oof(X, y, studies, n_splits=N_FOLDS, seed=RNG_SEED):
    """For the scatter figure: two OOF arrays.

    oof_solo[i] is non-NaN only for SDY569 subjects; it's their prediction
    from a model trained on SDY569 alone.

    oof_fed[i] is the federated-MLR prediction for every subject (same as
    federated_mlr_oof above).
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_solo = np.full_like(y, np.nan, dtype=float)
    oof_fed = np.full_like(y, np.nan, dtype=float)

    for tr, te in skf.split(X, studies):
        # SDY569 alone
        tr569 = tr[studies[tr] == "SDY569"]
        te569 = te[studies[te] == "SDY569"]
        if len(tr569) >= 2 and len(te569) >= 1:
            sc = MinMaxScaler().fit(X[tr569])
            m = LinearRegression().fit(sc.transform(X[tr569]), y[tr569])
            oof_solo[te569] = m.predict(sc.transform(X[te569]))

        # All-institutions federated
        local_models, scalers = {}, {}
        for s in sorted(set(studies[tr])):
            idx = tr[studies[tr] == s]
            if len(idx) < 2:
                continue
            sc = MinMaxScaler().fit(X[idx])
            scalers[s] = sc
            m = LinearRegression().fit(sc.transform(X[idx]), y[idx])
            local_models[s] = (m.coef_, m.intercept_, len(idx))
        coefs = np.stack([local_models[s][0] for s in local_models])
        ints = np.array([local_models[s][1] for s in local_models])
        sizes = np.array([local_models[s][2] for s in local_models])
        agg_coef = np.average(coefs, axis=0, weights=sizes)
        agg_int = np.average(ints, weights=sizes)
        for i in te:
            s = studies[i]
            if s not in scalers:
                continue
            xs = scalers[s].transform(X[i:i+1])[0]
            oof_fed[i] = agg_coef @ xs + agg_int

    return oof_solo, oof_fed


# Run for Panel A (the panel with all 4 studies including SDY569)
oof_solo, oof_fed = mlr_solo_vs_federated_oof(Xa, ya, studies_a)


In [ ]:
# Two-panel scatter
study_colors = {"SDY524": "#1f77b4", "SDY569": "#d62728",
                "SDY797": "#2ca02c", "SDY1737": "#9467bd"}

is569 = (studies_a == "SDY569")
mse_solo, r2_solo, n_solo = metrics(ya[is569], oof_solo[is569])
mse_fed, r2_fed, n_fed = metrics(ya, oof_fed)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5), constrained_layout=True)

ax = axes[0]
mask = is569 & ~np.isnan(oof_solo)
ax.scatter(ya[mask], oof_solo[mask], c=study_colors["SDY569"], s=65,
           edgecolor="white", linewidth=0.6,
           label=f"SDY569 (N={mask.sum()})")
if mask.sum() > 0:
    lo = min(ya[mask].min(), oof_solo[mask].min())
    hi = max(ya[mask].max(), oof_solo[mask].max())
    ax.plot([lo, hi], [lo, hi], "k--", alpha=0.4, label="y = x")
ax.set_xlabel("Observed log(C-peptide AUC)")
ax.set_ylabel("Predicted log(C-peptide AUC)")
ax.set_title(f"SDY569 alone  (N = {mask.sum()})\nMSE = {mse_solo:.3f}    R² = {r2_solo:+.3f}",
             fontweight="bold")
ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=0.25)

ax = axes[1]
for s in sorted(set(studies_a)):
    mask = (studies_a == s) & ~np.isnan(oof_fed)
    if mask.sum() == 0:
        continue
    ax.scatter(ya[mask], oof_fed[mask], c=study_colors.get(s, "grey"),
               s=65, edgecolor="white", linewidth=0.6, alpha=0.85,
               label=f"{s} (N={mask.sum()})")
all_mask = ~np.isnan(oof_fed)
if all_mask.sum() > 0:
    lo = min(ya[all_mask].min(), oof_fed[all_mask].min())
    hi = max(ya[all_mask].max(), oof_fed[all_mask].max())
    ax.plot([lo, hi], [lo, hi], "k--", alpha=0.4, label="y = x")
ax.set_xlabel("Observed log(C-peptide AUC)")
ax.set_ylabel("Predicted log(C-peptide AUC)")
ax.set_title(f"All institutions federated  (N = {all_mask.sum()})\nMSE = {mse_fed:.3f}    R² = {r2_fed:+.3f}",
             fontweight="bold")
ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=0.25)

fig.suptitle("Federated MLR — SDY569 alone vs all institutions (Panel A)",
             fontsize=13, fontweight="bold")
fig.savefig("figures/federated_mlr_solo_vs_fed.pdf", dpi=300)
fig.savefig("figures/federated_mlr_solo_vs_fed.png", dpi=220)
plt.show()


## 6. Talk-ready summary

The table below is the row of numbers a talk slide for this method needs.
One row per configuration: N, k, MSE, R², achieved power.


In [ ]:
print("Federated MLR (FedAvg-by-N) — talk summary")
print("=" * 78)
print(mlr_results.to_string(index=False))
print()
print(f"Saved to results/federated_mlr_results.csv")


## 7. What this notebook does *not* do

- **No pooled comparison.** Every fit in this notebook is per-institution.
  We never compute a `Lasso.fit(Xa, ya)` or any other model on the union
  of subject-level data. The "centralized would have done better" question
  is best addressed in a separate algorithm-verification notebook (where
  the comparison is to prove ADMM-Lasso converges to the centralised
  solution mathematically, not to advocate pooling in practice).
- **No CNN.** Convolutional networks are in their own notebook — the
  federated MLR story is cleaner without mixing in the deep model.
- **No LASSO regularisation.** When the feature set is small (Panel A:
  9 features; Panel B: 23) and N is modest, plain OLS is the right
  starting point. The `Federated_LASSO.ipynb` notebook addresses what
  changes when you want feature *selection* on top of federation.

## Suggested next reading

- `Federated_LASSO.ipynb` — federated L1 regression via ADMM consensus,
  with the median and FedAvg-by-N baselines for comparison.
- `Federated_RandomForest.ipynb` — federation via Union-of-Forests for
  non-linear methods.
